### Notebook for ETL of indicators for individual works  

#### There are indicators that relate to REFERENCING and indicators that relate to BEING CITED
#### For some indicators we know the "exogenous" total, for others only the "endogenous" values for the corpus proper

#### referencing indicators
-  publication year
-  present academic age of authors
-  number of references  
-  number of pages  
-  references/page
-  reference recency (Price index)  
-  copied reference count  
-  copied reference ratio  
-  self references  
-  self referencing rate (per reference)
-  cited sources FD
-  cited authors FD  
-  cited institutions FD    
-  cited topics FD  

#### being cited indicators
-  number of citations
-  fwci from oa  
-  fwci from corpus
-  disruption index    
-  hc status from oa  
-  hc status from corpus  
-  journal spectral rank  
-  institution spectral rank  
-  citer sources FD
-  citer authors FD  
-  citer institutions FD  
-  citer topics FD  



In [6]:
%run common_setup.ipynb

#### This cell extracts work-specific data and transforms it to the reference-related work indicators

In [7]:
class ReferencesETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def work_labels(self):
        sql = """
            CREATE OR REPLACE TABLE memory.work_labels AS
            -- ETL FOR ATTACHING AUTHOR Group/Class INFORMATION IN reference TABLE
            -- =======================================================
            WITH
                candidates_CTE AS
                (SELECT author_id,
                        "Group",
                        "Class",
                    FROM project.candidates
                    WHERE list_contains(['endogenous', 'exogenous', 'matched_difficult'], kind)
                ),
                works_authors_CTE AS
                (SELECT DISTINCT work_id,
                        author_id,
                        "Group",
                        "Class",
                    FROM project.authorships
                    LEFT JOIN candidates_CTE
                    USING (author_id)
                ),
                work_labeler_CTE AS
                (SELECT work_id,
                        list_sort(list_distinct(g)) AS g,
                        list_sort(list_distinct(c)) AS c,
                    FROM (SELECT work_id,
                            list("Group") FILTER ("Group" NOT NULL) AS g,
                            list("Class") FILTER ("Class" NOT NULL) AS c,
                            FROM works_authors_CTE
                            GROUP BY work_id
                        )
                )

            SELECT work_id,
                    CASE WHEN g = ['T'] THEN 'T' WHEN g = ['C'] THEN 'C' WHEN g = ['C', 'T'] THEN 'CT' ELSE 'X' END AS sample,
            FROM work_labeler_CTE           
            """
        self.db.sql(sql)  #.show()
        self.db.sql("SELECT * FROM memory.work_labels").show()
        return

    def references_per_page(self):
        sql = """
            CREATE OR REPLACE TABLE memory.references_per_page AS
            -- ETL FOR references_per_page
            -- ===========================
            SELECT id AS work_id,
                    publication_year,
                    referenced_works_count,
                    try_cast("biblio.last_page" AS INT) - try_cast("biblio.first_page" AS INT) AS page_count,
                    IF (page_count > 1, len(referenced_works)/page_count, NULL) AS references_per_page
                FROM project.raw
                ORDER BY references_per_page DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.references_per_page").show()
        return
        
    def copied_references(self):
        sql = """
            CREATE OR REPLACE TABLE memory.copied_references AS
            -- ETL FOR copied_references
            -- =========================
            WITH
            get_copied_CTE AS
                (SELECT r1.citer_id,
                        count(r2.cited_id) AS copied_count,
                        referenced_works_count AS total_count,
                    FROM project.citer_cited r1
                    LEFT JOIN project.citer_cited r2
                    ON r1.cited_id = r2.citer_id
                    LEFT JOIN project.raw
                    ON id = r1.citer_id
                    WHERE r1.cited_id = r2.cited_id
                    GROUP BY ALL
                )

            SELECT citer_id AS work_id,
                    100*copied_count/total_count AS copied_percent
            FROM get_copied_CTE
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.copied_references").show()
        return
  
    def self_references(self):
        sql = """
            CREATE OR REPLACE TABLE memory.self_reference_rate AS  
            -- ETL FOR self_reference_rate WITH FULL LIST OF REFERENCES
            -- ========================================================
            WITH
                referencing_author_list_CTE AS
                (SELECT DISTINCT work_id,
                        list(author_id) AS referencing_list
                    FROM project.authorships
                    GROUP BY work_id
                ),
                referenced_works_CTE AS
                (SELECT DISTINCT unnest(referenced_works) AS reference
                    FROM project.raw
                ),
                referenced_authors_CTE AS
                (SELECT DISTINCT reference,
                        list(DISTINCT author_id) AS referenced_list
                    FROM referenced_works_CTE
                    LEFT JOIN works.authorships
                    ON reference = work_id
                    GROUP BY reference
                ),
                self_reference_flag_CTE AS
                (SELECT DISTINCT id,
                        reference,
                        IF(len(list_intersect(sub.referencing_list, a.referenced_list)) > 0, 1, 0) AS isSelfReferenced
                    FROM (SELECT id,
                                unnest(referenced_works) AS reference,
                                referencing_list
                            FROM project.raw
                            LEFT JOIN referencing_author_list_CTE r
                            ON id = r.work_id
                        ) sub
                    LEFT JOIN referenced_authors_CTE a
                    USING (reference)
                )

            SELECT id AS work_id,
                    count(reference) AS reference_count,
                    sum(isSelfReferenced) AS reference_self_count,
                    sum(isSelfReferenced)/count(reference) AS self_reference_rate
            FROM self_reference_flag_CTE
            GROUP BY id
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.self_reference_rate").show()
        return
    
    def referenced_authors(self):
        sql = """
            CREATE OR REPLACE TABLE memory.referenced_authors AS  
            -- ETL FOR author_reference_rate WITH FULL LIST OF REFERENCES
            -- ========================================================    
            WITH
                referenced_works_CTE AS
                (SELECT DISTINCT id AS work_id,
                            referenced_works_count,
                        unnest(referenced_works) AS reference
                    FROM project.raw
                ),
                reference_authors_CTE AS
                (SELECT DISTINCT reference,
                        list(DISTINCT author_id) AS referenced_author_list
                    FROM referenced_works_CTE
                    LEFT JOIN works.authorships a
                    ON reference = a.work_id
                    GROUP BY reference
                ),
                all_referenced_authors_CTE AS
                (SELECT work_id,
                        referenced_works_count,
                        list(referenced_author_list) AS full_author_list
                    FROM (SELECT work_id,
                                referenced_works_count,
                                referenced_author_list
                            FROM referenced_works_CTE
                            LEFT JOIN reference_authors_CTE
                            USING (reference)
                        )
                    GROUP BY ALL
                ),
                author_counts_CTE AS
                (SELECT DISTINCT work_id,
                        referenced_works_count,
                        flatten(full_author_list) AS author_list_total,
                        len(flatten(full_author_list)) AS author_count_total,
                        list_unique(flatten(full_author_list)) AS author_count_unique,
                    FROM all_referenced_authors_CTE
                )

            SELECT DISTINCT ON (work_id)
                    *,
                    author_count/referenced_works_count AS most_referenced_author_frequency,
                    first_value(author_id) OVER (PARTITION BY work_id ORDER BY author_count DESC) AS most_referenced_author
            FROM
                (SELECT DISTINCT work_id,
                    author_id,
                    referenced_works_count,
                    count(author_id) AS author_count,
                    author_count_total,
                    author_count_unique
                FROM 
                    (SELECT work_id,
                            referenced_works_count,
                            author_count_total,
                            author_count_unique,
                            unnest(author_list_total) AS author_id,
                    FROM author_counts_CTE
                    )
                GROUP BY ALL
                )
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.referenced_authors").show()
        return
    
    def spectral_rank(self):
        sql = """  
            CREATE OR REPLACE TABLE memory.spectral_rank AS             
            -- ETL TO COLLECT source-pageRank AND institution-PageRank OF A PAPER'S REFERENCE LIST
            -- ===================================================================================
            WITH
                works_sources_CTE AS
                (SELECT id AS work_id,
                        "primary_location.source".display_name AS source_name,
                        "primary_location.source".id AS source_id,
                    FROM project.raw
                ),
                works_pageranks_CTE AS
                (SELECT work_id,
                        pageRank,
                        influence
                    FROM works_sources_CTE
                    LEFT JOIN project.pagerank_source
                    ON source_id = citer
                )

            SELECT citer_id AS work_id,
                    sum(pagerank) AS pagerank_total,
                    avg(pagerank) AS pagerank_avg,
                    sum(influence) AS influence_total,
                    avg(influence) AS influence_avg,                    
            FROM project.citer_cited
            LEFT JOIN works_pageranks_CTE p
            ON p.work_id = cited_id
            GROUP BY citer_id
        """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.spectral_rank").show()
        return
    
    def reference_combiner(self):
        self.db.sql("SHOW ALL TABLES").show()
        sql = "CREATE OR REPLACE TABLE project.references_work_summary AS (SELECT * FROM memory.work_labels\n"
        for tab in ['references_per_page', 'copied_references', 'self_reference_rate', 'referenced_authors', 'spectral_rank']:
            sql = f"{sql}LEFT JOIN (SELECT * FROM memory.{tab}) USING (work_id)\n"
        sql = f"{sql})"

        self.db.sql(sql)
        df = self.db.sql("SELECT * FROM project.references_work_summary").df()
        print(f'{df.shape = }\n{df.head(16)}')
        return

#### This cell extracts work-specific data and transforms it to the citation-related work indicators

In [ ]:
class CitationsETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def work_labels(self):
        sql = """
            CREATE OR REPLACE TABLE memory.work_labels AS
            -- ETL FOR ATTACHING AUTHOR Group/Class INFORMATION IN reference TABLE
            -- =======================================================
            WITH
                candidates_CTE AS
                (SELECT author_id,
                        "Group",
                        "Class",
                    FROM project.candidates
                    WHERE list_contains(['endogenous', 'exogenous', 'matched_difficult'], kind)
                ),
                works_authors_CTE AS
                (SELECT DISTINCT work_id,
                        author_id,
                        "Group",
                        "Class",
                    FROM project.authorships
                    LEFT JOIN candidates_CTE
                    USING (author_id)
                ),
                work_labeler_CTE AS
                (SELECT work_id,
                        list_sort(list_distinct(g)) AS g,
                        list_sort(list_distinct(c)) AS c,
                    FROM (SELECT work_id,
                            list("Group") FILTER ("Group" NOT NULL) AS g,
                            list("Class") FILTER ("Class" NOT NULL) AS c,
                            FROM works_authors_CTE
                            GROUP BY work_id
                        )
                )

            SELECT work_id,
                    CASE WHEN g = ['T'] THEN 'T' WHEN g = ['C'] THEN 'C' WHEN g = ['C', 'T'] THEN 'CT' ELSE 'X' END AS sample,
            FROM work_labeler_CTE           
            """
        self.db.sql(sql)  #.show()
        self.db.sql("SELECT * FROM memory.work_labels").show()
        return    
    
    def citation_count(self):
        sql = """
            CREATE OR REPLACE TABLE memory.citation_count AS  
            -- ETL FOR citation_count
            -- ======================
            SELECT cited_id AS work_id,
                    cited_by_count,
                    count(citer_id) AS cited_by_count_endogenous
            FROM project.citer_cited
            LEFT JOIN project.raw
            ON id = cited_id
            -- WHERE len(authorships) > 0
            GROUP BY ALL
            ORDER BY cited_by_count_endogenous DESC           
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.citation_count").show()
        return
    
    def fwci(self):
        sql = """
            CREATE OR REPLACE TABLE memory.fwci AS
            WITH
            get_fwci_endogenous_CTE AS
                (SELECT DISTINCT cited_id,
                                count(citer_id) OVER (PARTITION BY cited_id)/
                                    (count(citer_id) OVER (PARTITION BY cited_year)/
                                    count(DISTINCT citer_id) OVER (PARTITION BY cited_year)) AS fwci_endogenous,
                FROM project.citer_cited
                ORDER BY fwci_endogenous DESC
                )
            SELECT id AS work_id,
                    fwci,
                    fwci_endogenous
                FROM project.raw
                LEFT JOIN get_fwci_endogenous_CTE
                ON id = cited_id
            ORDER BY fwci DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.fwci").show()
        return
    
    def highly_cited(self):
        sql = """
            CREATE OR REPLACE TABLE memory.highly_cited AS  
            -- ETL FOR highly_cited_work
            -- =========================
            WITH
            citation_counts_CTE AS
                (SELECT DISTINCT cited_by_count,
                        count(citer_id) OVER (PARTITION BY cited_id) AS cited_by_count_endogenous,
                        cited_id,
                        cited_year,
                FROM project.citer_cited
                LEFT JOIN project.raw
                ON id = cited_id
                -- ORDER BY cited_by_count_endogenous DESC
                ),
            quantiles_CTE AS
                (SELECT DISTINCT cited_id,
                        cited_year,
                        quantile_disc(cited_by_count, 0.99) OVER (PARTITION BY cited_year) AS quant,
                        quantile_disc(cited_by_count_endogenous, 0.99) OVER (PARTITION BY cited_year) AS quant_endogenous
                FROM citation_counts_CTE
                -- ORDER BY cited_year DESC
                ),
            highly_cited_CTE AS
                (SELECT DISTINCT cc.cited_id,
                        IF (cited_by_count_endogenous - quant_endogenous > 0, true, false) AS isHighlyCitedEndogenous,
                        IF (cited_by_count - quant > 0, true, false) AS isHighlyCited
                FROM citation_counts_CTE cc
                LEFT JOIN quantiles_CTE
                USING (cited_year)
                )
            SELECT cited_id AS work_id,
                    isHighlyCited,  
                    isHighlyCitedEndogenous
            FROM highly_cited_CTE

            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.highly_cited").show()
        return
    
    def disruption_index(self):
        sql = """
            CREATE OR REPLACE TABLE memory.disruption_index AS  
            -- ETL FOR disruption_index
            -- ========================
            WITH
            extract_work_CTE AS
                (SELECT id AS work_id,
                        fwci,
                        unnest(referenced_works) AS referenced_work
                FROM project.raw
                ),
            extract_disruption_index_CTE AS
                (SELECT work_id,
                        100*r1.fwci/avg(r2.fwci) OVER (PARTITION BY work_id) AS disruption_index
                FROM extract_work_CTE r1
                LEFT JOIN project.raw r2
                ON referenced_work = r2.id
                ORDER BY disruption_index DESC
                )

            SELECT DISTINCT work_id,
                    disruption_index
                FROM extract_disruption_index_CTE
                WHERE disruption_index NOT NULL AND disruption_index != 'NaN' AND disruption_index != 'Infinity'
                ORDER BY disruption_index DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.disruption_index").show()
        return

    def topic_indicators(self):
        sql = """
            CREATE OR REPLACE TABLE memory.topic_indicators AS  
            -- ETL FOR topic_indicators
            -- =======================
            WITH
                topics_CTE AS
                (SELECT work_id,
                        domain_id[-1:] AS domain_id,
                        field_id[-2:] AS field_id,
                        subfield_id[-4:] AS subfield_id
                    FROM project.topics
                ),
                citer_cited_topics_CTE AS
                (SELECT citer_id,
                        t1.domain_id AS domain_id_citer,
                        t1.field_id AS field_id_citer,
                        t1.subfield_id AS subfield_id_citer,
                        cited_id,
                        t2.domain_id AS domain_id_cited,
                        t2.field_id AS field_id_cited,
                        t2.subfield_id AS subfield_id_cited,
                    FROM project.citer_cited
                    LEFT JOIN topics_CTE as t1
                    ON citer_id = t1.work_id
                    LEFT JOIN topics_CTE t2
                    ON cited_id = t2.work_id
                ),
                cited_subfield_list_CTE AS
                (SELECT citer_id,
                        list(subfield_id_cited) AS cited_subfield_list
                    FROM citer_cited_topics_CTE
                    GROUP BY citer_id
                ),
                citer_subfield_list_CTE AS
                (SELECT cited_id,
                        list(subfield_id_citer) AS citer_subfield_list
                    FROM citer_cited_topics_CTE
                    GROUP BY cited_id
                ),
                citer_topic_CTE AS
                (SELECT citer_id,
                        subfield_id_citer,
                        list_distinct(cited_subfield_list) AS cited_topics,
                        len(list_distinct(cited_subfield_list)) AS cited_topics_count
                    FROM citer_cited_topics_CTE cct
                    LEFT JOIN cited_subfield_list_CTE
                    USING (citer_id)
                    GROUP BY ALL
                    ORDER BY cited_topics_count DESC
                ),
                cited_topic_CTE AS
                (SELECT cited_id,
                        subfield_id_cited,
                        list_distinct(citer_subfield_list) AS citer_topics,
                        len(list_distinct(citer_subfield_list)) AS citer_topics_count
                    FROM citer_cited_topics_CTE cct
                    LEFT JOIN citer_subfield_list_CTE
                    USING (cited_id)
                    GROUP BY ALL
                    ORDER BY citer_topics_count DESC
                )

            SELECT citer_id AS work_id,
                    subfield_id_citer AS subfield_id,
                    cited_topics,
                    cited_topics_count,
                    citer_topics,
                    citer_topics_count
            FROM citer_topic_CTE
            LEFT JOIN cited_topic_CTE
            ON citer_id = cited_id
            ORDER BY citer_topics_count DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.topic_indicators").show()
        return
    
    def spectral_ranks(self):
        sql = """
            CREATE OR REPLACE TABLE memory.spectral_ranks AS
            -- ETL FOR pagerank AND influence
            -- ==============================
            WITH
                select_work_journal_CTE AS
                (SELECT id AS work_id,
                        "primary_location.source".id AS source_id,
                        pagerank,
                        influence,
                    FROM project.raw
                    LEFT JOIN project.pagerank_source
                    ON "primary_location.source".id = citer
                ),
                select_work_institutions_CTE AS
                (SELECT DISTINCT work_id,
                        institution_id,
                        count(DISTINCT institution_id) OVER (PARTITION BY work_id) AS institution_count,
                        pagerank,
                        influence
                    FROM project.authorships
                    LEFT JOIN project.pagerank_institution
                    ON institution_id = citer 
                ),
                select_work_institution_CTE AS
                (SELECT work_id,
                        sum(pagerank)/institution_count AS pagerank_institution,
                        sum(influence)/institution_count AS influence_institution
                    FROM select_work_institutions_CTE
                    GROUP BY work_id, institution_count
                )

            SELECT work_id,
                    100*pagerank AS pagerank_source,
                    100*influence AS influence_source,
                    100*pagerank_institution AS pagerank_institution,
                    100*influence_institution AS influence_institution
            FROM select_work_journal_CTE
            LEFT JOIN select_work_institution_CTE
            USING (work_id)
            ORDER BY pagerank_institution DESC, pagerank_source DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.spectral_ranks").show()
        return

    def citation_combiner(self):
        self.db.sql("SHOW ALL TABLES").show()
        sql = "CREATE OR REPLACE TABLE project.citations_work_summary AS (SELECT * FROM memory.work_labels\n"
        for tab in ['citation_count', 'fwci', 'highly_cited', 'disruption_index', 'topic_indicators', 'spectral_ranks']:
            sql = f"{sql}LEFT JOIN (SELECT * FROM memory.{tab}) USING (work_id)\n"
        sql = f"{sql})"
        self.db.sql(sql)
        df = self.db.sql("SELECT * FROM project.citations_work_summary").df()
        print(f'{df.shape = }\n{df.head(16)}')
        return
            


In [9]:
def main():

    # retl = ReferencesETL()
    # retl.work_labels()
    # retl.references_per_page()
    # retl.copied_references()
    # retl.self_references()
    # retl.referenced_authors()
    # retl.spectral_rank()
    # retl.reference_combiner()
    # retl.db.close()

    cetl = CitationsETL()
    cetl.work_labels()
    cetl.citation_count()
    cetl.fwci()
    cetl.highly_cited()
    cetl.disruption_index()
    cetl.topic_indicators()
    cetl.spectral_ranks()
    cetl.citation_combiner()
    cetl.db.close()
        
    return

In [10]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ authors  │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ authors  │ main    │ trace                │ [id, works_count, …  │ [VARCHAR, BIGINT, BIGINT, BIGINT, D…  │ false     │
│ backup   │ main    │ authors              │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authors_full         │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authorshi